# 02 — Baseline: CatBoost 5-fold OOF
Establish the fixed CV folds and the number to beat. CatBoost is the primary model: it handles the 50% NaN in `eog_burst_index` natively and optimizes macro-F1 directly (`eval_metric='TotalF1:average=Macro'`). We also run a RandomForest as a cross-check.

In [1]:
# --- Shared toolbox (identical across notebooks; see build_notebooks.py) ---
import warnings, json
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
N_FOLDS = 5
CLASS_NAMES = {0: "Wake", 1: "Light", 2: "Deep", 3: "REM"}
CLASSES = np.array([0, 1, 2, 3])
EOG = "eog_burst_index"            # the only column with missing values (~50%)

RAW_FEATURES = [
    "eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
    "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power", "eeg_spectral_entropy",
    "eeg_spindle_density", "eeg_kcomplex_rate", "emg_chin_tone", "emg_tone_variance",
    "eog_movement_density", "eog_amplitude", "heart_rate_mean", "heart_rate_variability",
    "respiration_rate", "respiration_variability", "spo2_mean", "body_movement_index",
    EOG,
]
POWER = ["eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
         "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power"]

HERE = Path.cwd()
ART = HERE / "artifacts"; ART.mkdir(exist_ok=True)
SUB = HERE / "submissions"; SUB.mkdir(exist_ok=True)


def load_data():
    """Return (train_df, test_df). Features kept as-is (NaN preserved)."""
    tr = pd.read_csv(HERE / "train.csv")
    te = pd.read_csv(HERE / "test.csv")
    return tr, te


def macro_f1(y_true, y_pred):
    """Competition metric: macro-averaged F1 over the 4 classes."""
    return f1_score(y_true, y_pred, average="macro")


def per_class_f1(y_true, y_pred):
    f = f1_score(y_true, y_pred, average=None, labels=CLASSES)
    return {CLASS_NAMES[c]: round(float(f[i]), 4) for i, c in enumerate(CLASSES)}


def softplus(x):
    """Numerically stable log(1+exp(x)); strictly positive and monotonic.
    Used to turn z-scored band powers (~50% negative) into positive magnitudes
    so band ratios are well-defined instead of dividing by near-zero."""
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _aligned_proba(model, X):
    """predict_proba with columns aligned to CLASSES = [0,1,2,3]."""
    p = model.predict_proba(X)
    cls = list(np.asarray(model.classes_))
    idx = [cls.index(c) for c in CLASSES]
    return p[:, idx]


def run_oof(make_model, X, y, X_test, folds, needs_impute=False, use_eval_set=False):
    """Out-of-fold training under fixed folds.

    Returns (oof, test_p, oof_macro, fold_scores):
      oof     : (n_train, 4) out-of-fold probabilities (each row predicted once)
      test_p  : (n_test, 4) test probabilities, averaged over the 5 fold-models
      oof_macro: global macro-F1 over the assembled OOF matrix (primary metric)

    Two model families, identical folds:
      - CatBoost (needs_impute=False): NaN passed through natively.
      - sklearn trees (needs_impute=True): add EOG-missing flag + fill EOG NaN
        with the TRAIN-FOLD median (fit on train fold only -> no leakage)."""
    n = len(y)
    oof = np.zeros((n, len(CLASSES)))
    test_p = np.zeros((len(X_test), len(CLASSES)))
    fold_scores = []
    for tr_idx, va_idx in folds:
        Xtr, Xva, Xte = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy(), X_test.copy()
        ytr, yva = y[tr_idx], y[va_idx]
        if needs_impute:
            med = Xtr[EOG].median()
            for d in (Xtr, Xva, Xte):
                if EOG + "_missing" not in d.columns:
                    d[EOG + "_missing"] = d[EOG].isna().astype("int8")
                d[EOG] = d[EOG].fillna(med)
            assert not Xtr.isna().any().any(), "NaN remained after impute"
        model = make_model()
        if use_eval_set:
            model.fit(Xtr, ytr, eval_set=(Xva, yva))
        else:
            model.fit(Xtr, ytr)
        oof[va_idx] = _aligned_proba(model, Xva)
        test_p += _aligned_proba(model, Xte) / len(folds)
        fold_scores.append(macro_f1(yva, oof[va_idx].argmax(1)))
    oof_macro = macro_f1(y, oof.argmax(1))
    return oof, test_p, oof_macro, fold_scores


def load_folds():
    """Load the fixed StratifiedKFold split saved by 02_baseline."""
    d = np.load(ART / "folds.npz", allow_pickle=True)
    return [(d[f"tr{i}"], d[f"va{i}"]) for i in range(N_FOLDS)]


In [2]:
def log_result(step, model, feature_set, oof_macro, pcf, notes=""):
    """Write one row to results_log.csv. Idempotent per (step, model): re-running
    a notebook replaces its own row rather than duplicating it."""
    import csv
    path = HERE / "results_log.csv"
    header = ["step", "model", "feature_set", "oof_macro_f1",
              "f1_Wake", "f1_Light", "f1_Deep", "f1_REM", "notes"]
    rows = []
    if path.exists():
        with open(path, newline="") as fh:
            data = list(csv.reader(fh))
        if data and data[0] == header:
            rows = [r for r in data[1:] if not (len(r) >= 2 and r[0] == step and r[1] == model)]
    row = [step, model, feature_set, round(float(oof_macro), 5),
           pcf["Wake"], pcf["Light"], pcf["Deep"], pcf["REM"], notes]
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header); w.writerows(rows); w.writerow(row)
    print("logged:", step, model, "OOF macro-F1 =", round(float(oof_macro), 5))


## Fix and save the folds
`StratifiedKFold(5, shuffle=True, random_state=42)`. Saved to `artifacts/folds.npz` so **every** later notebook uses byte-identical folds — the precondition for valid OOF blending.

In [3]:
from sklearn.model_selection import StratifiedKFold
(HERE / "results_log.csv").unlink(missing_ok=True)   # fresh evidence log per full run
train, test = load_data()
y = train["sleep_stage"].values.astype(int)
test_ids = test["id"].values
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = [(tr, va) for tr, va in skf.split(np.zeros(len(y)), y)]
np.savez(ART / "folds.npz", **{f"tr{i}": tr for i, (tr, _) in enumerate(folds)},
         **{f"va{i}": va for i, (_, va) in enumerate(folds)})
np.save(ART / "y_train.npy", y)
np.save(ART / "test_ids.npy", test_ids)
print("folds saved; sizes:", [len(va) for _, va in folds])

folds saved; sizes: [1800, 1800, 1800, 1800, 1800]


## CatBoost baseline (raw 21 features + missing flag)
CatBoost keeps the NaN; we add an explicit `eog_burst_missing` flag on top (cheap, near-certain help).

In [4]:
from catboost import CatBoostClassifier

def build_base(df):
    X = df[RAW_FEATURES].copy()
    X[EOG + "_missing"] = df[EOG].isna().astype("int8")
    return X

Xtr_base, Xte_base = build_base(train), build_base(test)

def make_catboost():
    return CatBoostClassifier(
        loss_function="MultiClass", eval_metric="TotalF1:average=Macro",
        iterations=3000, learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
        random_seed=SEED, od_type="Iter", od_wait=150, use_best_model=True,
        thread_count=-1, allow_writing_files=False, verbose=False)

cat_oof, cat_test, cat_macro, cat_folds = run_oof(
    make_catboost, Xtr_base, y, Xte_base, folds, needs_impute=False, use_eval_set=True)
print("CatBoost baseline OOF macro-F1:", round(cat_macro, 5))
print("per-fold:", [round(s, 4) for s in cat_folds],
      "| mean %.4f +/- %.4f" % (np.mean(cat_folds), np.std(cat_folds)))
print("per-class F1:", per_class_f1(y, cat_oof.argmax(1)))

CatBoost baseline OOF macro-F1: 0.82494
per-fold: [0.8272, 0.8119, 0.8308, 0.8224, 0.8322] | mean 0.8249 +/- 0.0073
per-class F1: {'Wake': 0.8522, 'Light': 0.8473, 'Deep': 0.7712, 'REM': 0.8291}


In [5]:
print(confusion_matrix(y, cat_oof.argmax(1)))
print(classification_report(y, cat_oof.argmax(1),
      target_names=[CLASS_NAMES[c] for c in CLASSES]))
np.save(ART / "cat_base_oof.npy", cat_oof)
np.save(ART / "cat_base_test.npy", cat_test)
log_result("02_baseline", "catboost", "raw21+missing", cat_macro,
           per_class_f1(y, cat_oof.argmax(1)), "primary baseline")

[[1689   71  157   84]
 [  60 2094  163  125]
 [ 124  199 1699  215]
 [  90  137  150 1943]]
              precision    recall  f1-score   support

        Wake       0.86      0.84      0.85      2001
       Light       0.84      0.86      0.85      2442
        Deep       0.78      0.76      0.77      2237
         REM       0.82      0.84      0.83      2320

    accuracy                           0.82      9000
   macro avg       0.83      0.82      0.82      9000
weighted avg       0.82      0.82      0.82      9000

logged: 02_baseline catboost OOF macro-F1 = 0.82494


## RandomForest cross-check
Same folds, but sklearn trees can't take NaN → impute EOG with the train-fold median + keep the missing flag (handled in `run_oof`).

In [6]:
from sklearn.ensemble import RandomForestClassifier
def make_rf():
    return RandomForestClassifier(n_estimators=600, max_features="sqrt",
        min_samples_leaf=2, class_weight="balanced", n_jobs=-1, random_state=SEED)
rf_oof, rf_test, rf_macro, rf_folds = run_oof(
    make_rf, Xtr_base, y, Xte_base, folds, needs_impute=True)
print("RandomForest OOF macro-F1:", round(rf_macro, 5))
print("per-class F1:", per_class_f1(y, rf_oof.argmax(1)))
log_result("02_baseline", "randomforest", "raw21+missing", rf_macro,
           per_class_f1(y, rf_oof.argmax(1)), "cross-check")

RandomForest OOF macro-F1: 0.79899
per-class F1: {'Wake': 0.8204, 'Light': 0.8302, 'Deep': 0.738, 'REM': 0.8074}
logged: 02_baseline randomforest OOF macro-F1 = 0.79899


## First end-to-end submission (`sub00`)
Fold-bagged CatBoost test probabilities, plain argmax. Proves the submission path works before we optimize.

In [7]:
pred = cat_test.argmax(1)
sub = pd.DataFrame({"id": test_ids, "sleep_stage": pred.astype(int)})
assert sub.shape == (5000, 2) and sub["sleep_stage"].isin(CLASSES).all()
sub.to_csv(SUB / "sub00_catboost_baseline.csv", index=False)
print("wrote", SUB / "sub00_catboost_baseline.csv")
print(sub["sleep_stage"].value_counts().sort_index().to_dict())

wrote C:\Users\rasul\Documents\workspace_ws\yessenov_data_lab_program\Day9\submissions\sub00_catboost_baseline.csv
{0: 1103, 1: 1309, 2: 1286, 3: 1302}
